* Modelo com constraint e normalização de price por sku 
* features de lag recursivas
* treinado separadamente para cada combinação sku-channel

In [4]:
"""
RetailCo Demand Model — Diagnostic Retrain
============================================
Business Case: Demand Model Optimization at RetailCo

Purpose
-------
1. Load the raw data (Table 1 - External Variables, Table 2 - Sell In).
2. Engineer features exactly as described in Table 4 (Feature Engineering Rules).
3. Train the BASELINE model using the current parameters (Table 3).
4. Train an IMPROVED model that targets the two root causes identified in the
   diagnostic phase:
     (a) Overfitting from excessive model capacity + zero regularization on a
         small weekly sample (156 weeks / SKU-channel).
     (b) Wrong-signed price elasticity (positive SHAP contribution for price),
         fixed with a native monotonic constraint on Price_per_kg_USD.
5. Evaluate both models with Accuracy = 1 - MAPE, per channel and overall.
6. Validate the Problem 2 fix with real SHAP values (sign consistency for
   price, and Promotion vs Advertising attribution share).
7. Root-cause follow-up: since price sign consistency alone doesn't fully
   resolve after the monotonic constraint, test a normalized Price_Index
   (price relative to each SKU's own average) to remove the cross-product
   price-scale confound created by pooling all 5 products into one model
   per channel.

Requirements
------------
pip install lightgbm shap pandas numpy scikit-learn matplotlib openpyxl

Usage
-----
1. Edit DATA_PATH below (in the CONFIG section) to point to your Excel file.
2. Run either:
   - In a terminal:        python retrain_demand_model.py
   - In Jupyter/Colab:     run all cells, then call main() in the last cell
"""

# ---------------------------------------------------------------------------
# 1. LIBRARIES
# ---------------------------------------------------------------------------

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
# 2. CONFIG
# ---------------------------------------------------------------------------

# >>> EDIT THESE TWO LINES <<<
# Path to the Business Case Data Set Excel file on your machine.
DATA_PATH = "Business_Case_Data_Set.xlsx"
# Folder where results (CSV + plots) will be saved. Created automatically.
OUTDIR = "output"

FEATURE_COLS = [
    "Price_per_kg_USD",
    "Numeric_Distribution",
    "Weighted_Distribution",
    "Advertising_Investment_USD",
    "Promotion_Investment_USD",
    "Promo_Flag",
    "Lag_1w",
    "Lag_4w",
    "Rolling_Mean_4w",
    "Rolling_Std_4w",
    "Price_Change_Pct",
    "Month_Sin",
    "Month_Cos",
    "Holiday_Flag",
    "Interaction_Price_Promo",
]

TARGET_COL = "Sell_In_Tons"

# Feature set for the price-normalization variant (used later by
# add_price_index / IMPROVED_V3_PARAMS): identical to FEATURE_COLS, but
# Price_per_kg_USD -> Price_Index and Interaction_Price_Promo ->
# Interaction_PriceIndex_Promo.
FEATURE_COLS_NORM = [
    {"Price_per_kg_USD": "Price_Index", "Interaction_Price_Promo": "Interaction_PriceIndex_Promo"}.get(c, c)
    for c in FEATURE_COLS
]

HOLDOUT_QUANTILE = 0.85  # last ~15% of weeks per SKU-channel used as test set

CATEGORY_MAP = {
    "Natural Juice 1L": "Beverages",
    "Flavored Water 500ml": "Beverages",
    "Energy Drink 350ml": "Beverages",
    "Whole Grain Crackers 200g": "Snacks",
    "Cereal Bar 50g": "Snacks",
}

# Params exactly as reported in Table 3 - Model Parameters
BASELINE_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    num_leaves=256,
    min_child_samples=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.0,
    reg_lambda=0.0,
    min_split_gain=0.0,
    boosting_type="gbdt",
    objective="regression",
    metric="mape",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# Regularized + monotonic-constrained candidate.
# monotone_constraints: -1 forces Price_per_kg_USD to have a non-positive
# effect on predicted volume (correct economic sign). All other features
# are left unconstrained (0).
IMPROVED_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_split_gain=0.0,
    boosting_type="gbdt",
    objective="regression",
    metric="mape",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    monotone_constraints=[
        -1 if col == "Price_per_kg_USD" else 0 for col in FEATURE_COLS
    ],
    monotone_constraints_method="advanced",
)

# Diagnostic variant: same as IMPROVED_PARAMS, but ALSO constrains
# Interaction_Price_Promo (= Price_per_kg_USD * Promo_Flag) to be
# non-increasing in price. Tests the hypothesis that the positive price
# SHAP signal is "leaking" through the unconstrained interaction term.
IMPROVED_V2_PARAMS = dict(
    IMPROVED_PARAMS,
    monotone_constraints=[
        -1 if col in ("Price_per_kg_USD", "Interaction_Price_Promo") else 0
        for col in FEATURE_COLS
    ],
)

# V3: same regularization as IMPROVED_PARAMS, but built for FEATURE_COLS_NORM
# (raw price replaced with the SKU-relative Price_Index). Constrains
# Price_Index to be non-increasing, same economic logic as before.
IMPROVED_V3_PARAMS = dict(
    IMPROVED_PARAMS,
    monotone_constraints=[
        -1 if col == "Price_Index" else 0 for col in FEATURE_COLS_NORM
    ],
)

# SKU-level variant: one model PER Product x Channel combo, instead of
# pooling all 5 products into one model per channel. Each model now sees
# only ~130 training weeks (vs ~650 pooled), so the tree is deliberately
# simpler than IMPROVED_PARAMS to avoid overfitting on less data. No
# monotonic constraint here -- this test isolates the effect of NOT
# pooling on raw accuracy, independent of the Problem 2 fix.
SKU_LEVEL_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=4,
    num_leaves=15,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_split_gain=0.0,
    boosting_type="gbdt",
    objective="regression",
    metric="mape",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)


# ---------------------------------------------------------------------------
# 3. LOAD DATA
# ---------------------------------------------------------------------------

def load_data(path: str) -> pd.DataFrame:
    """
    Loads Table 1 (External Variables) and Table 2 (Sell In) from the
    Business Case Data Set and merges them on Week/Product/Channel.
    Returns the raw merged dataframe, BEFORE any feature engineering.
    """
    xl = pd.ExcelFile(path)
    ext = xl.parse("Table 1 - External Variables")
    actual = xl.parse("Table 2 - Sell In")

    df = ext.merge(actual, on=["Week", "Product", "Channel"], how="inner")
    df["Week"] = pd.to_datetime(df["Week"])
    df["Category"] = df["Product"].map(CATEGORY_MAP)
    df = df.sort_values(["Product", "Channel", "Week"]).reset_index(drop=True)
    return df


# ---------------------------------------------------------------------------
# 4. FEATURE ENGINEERING (mirrors Table 4 - Feature Engineering Rules)
# ---------------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies the 10 feature engineering rules from Table 4 to the raw
    merged dataframe. Returns a dataframe with all FEATURE_COLS populated,
    with rows containing NaN (e.g. the first weeks of each SKU-channel,
    which have no lag history) dropped.
    """
    df = df.copy()

    # FE_06 Promo_Flag: 1 if Promotion_Type != None, else 0
    df["Promo_Flag"] = df["Promotion_Type"].notna().astype(int)

    g_target = df.groupby(["Product", "Channel"])[TARGET_COL]

    # FE_01 / FE_02: lags of Sell-In volume
    df["Lag_1w"] = g_target.shift(1)
    df["Lag_4w"] = g_target.shift(4)

    # FE_03 / FE_04: rolling mean/std, computed on the SHIFTED series
    # so the current week's own volume is never used to predict itself
    shifted = g_target.shift(1)
    grouped_shifted = shifted.groupby([df["Product"], df["Channel"]])
    df["Rolling_Mean_4w"] = grouped_shifted.rolling(4).mean().reset_index(
        level=[0, 1], drop=True
    )
    df["Rolling_Std_4w"] = grouped_shifted.rolling(4).std().reset_index(
        level=[0, 1], drop=True
    )

    # FE_05: week-over-week % change in price
    df["Price_Change_Pct"] = df.groupby(["Product", "Channel"])[
        "Price_per_kg_USD"
    ].pct_change()

    # FE_07 / FE_08: cyclical month encoding
    month = df["Week"].dt.month
    df["Month_Sin"] = np.sin(2 * np.pi * month / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * month / 12)

    # FE_09: Holiday_Flag
    # ASSUMPTION: no explicit holiday calendar table was provided in the
    # dataset, so this defaults to 0. If you have a real calendar, merge a
    # Holiday_Flag column keyed by Week BEFORE calling this function, and
    # remove the line below.
    df["Holiday_Flag"] = 0

    # FE_10: interaction term between price and promotion
    df["Interaction_Price_Promo"] = df["Price_per_kg_USD"] * df["Promo_Flag"]

    df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
    return df


def time_split(df: pd.DataFrame):
    """
    Time-based train/test split, computed independently per SKU-channel so
    every series contributes its most recent ~15% of weeks to the test set.
    This avoids leakage (no future data used to predict the past).
    """
    cutoff = df.groupby(["Product", "Channel"])["Week"].transform(
        lambda x: x.quantile(HOLDOUT_QUANTILE, interpolation="nearest")
    )
    train = df[df["Week"] <= cutoff].copy()
    test = df[df["Week"] > cutoff].copy()
    return train, test


def add_price_index(train: pd.DataFrame, test: pd.DataFrame):
    """
    Root-cause fix: models are trained pooling ALL 5 products within a
    channel, but products sit on very different price scales (e.g.
    Flavored Water ~$1.80/kg vs Cereal Bar ~$12/kg). SHAP explains a
    feature relative to the POPULATION average, so a structurally cheap
    product like Natural Juice can show a "wrong-signed" price SHAP even
    when the model's actual price-volume relationship is correct for
    that SKU (confirmed via partial dependence).

    Fix: replace the raw price with a self-referential index --
        Price_Index = Price_per_kg_USD / (product's own average price)
    so 1.0 always means "this SKU's own typical price", regardless of
    the SKU's absolute price level. The per-product average is computed
    from the TRAINING set only, then applied to both train and test, to
    avoid leakage.
    """
    product_mean_price = train.groupby("Product")["Price_per_kg_USD"].mean()

    train = train.copy()
    test = test.copy()
    train["Price_Index"] = train["Price_per_kg_USD"] / train["Product"].map(product_mean_price)
    test["Price_Index"] = test["Price_per_kg_USD"] / test["Product"].map(product_mean_price)

    train["Interaction_PriceIndex_Promo"] = train["Price_Index"] * train["Promo_Flag"]
    test["Interaction_PriceIndex_Promo"] = test["Price_Index"] * test["Promo_Flag"]

    return train, test


# ---------------------------------------------------------------------------
# 5. MODEL TRAINING
# ---------------------------------------------------------------------------

def train_model(X: pd.DataFrame, y: pd.Series, params: dict) -> lgb.LGBMRegressor:
    """Fits a single LightGBM regressor with the given parameter set."""
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model


def train_all_channels(train: pd.DataFrame, params: dict, feature_cols: list = None) -> dict:
    """
    Trains one model per channel (mirrors RetailCo's current slicing
    approach: one model per channel, sharing the same parameter set).
    Returns {channel: fitted_model}.
    """
    feature_cols = feature_cols or FEATURE_COLS
    models = {}
    for channel in train["Channel"].unique():
        sub = train[train["Channel"] == channel]
        X, y = sub[feature_cols], sub[TARGET_COL]
        models[channel] = train_model(X, y, params)
    return models


def train_all_sku_channels(train: pd.DataFrame, params: dict, feature_cols: list = None) -> dict:
    """
    Trains one model PER Product x Channel combination -- no pooling across
    products. Returns {(product, channel): fitted_model}.

    Trade-off vs train_all_channels(): each model sees far less data
    (~130 weeks vs ~650 pooled), but by construction it can never confuse
    price scale across products (no Price_Index normalization needed --
    the model only ever sees ONE product's price range).
    """
    feature_cols = feature_cols or FEATURE_COLS
    models = {}
    for (product, channel), sub in train.groupby(["Product", "Channel"]):
        if len(sub) < 15:
            continue  # not enough history to train a meaningful model
        X, y = sub[feature_cols], sub[TARGET_COL]
        models[(product, channel)] = train_model(X, y, params)
    return models


# ---------------------------------------------------------------------------
# 6. MODEL EVALUATION — Accuracy = 1 - MAPE
# ---------------------------------------------------------------------------

def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Computes MAPE and Accuracy (1 - MAPE), matching the metric definition
    used throughout the business case (Table 3: metric = 'mape';
    Problem Statement: Model Accuracy = 1 - MAPE).
    """
    mape = float(np.mean(np.abs(y_true - y_pred) / y_true))
    return {"MAPE": mape, "Accuracy": 1 - mape}


def evaluate_all_channels(models: dict, test: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """
    Evaluates a set of per-channel models against the test set and returns
    a per-channel + overall (volume-weighted) accuracy table.
    """
    feature_cols = feature_cols or FEATURE_COLS
    rows = []
    for channel, model in models.items():
        sub = test[test["Channel"] == channel]
        if len(sub) == 0:
            continue
        X, y_true = sub[feature_cols], sub[TARGET_COL].values
        y_pred = model.predict(X)
        metrics = evaluate_model(y_true, y_pred)
        rows.append({"Channel": channel, "n_test": len(sub), **metrics})

    result = pd.DataFrame(rows)
    overall_mape = np.average(result["MAPE"], weights=result["n_test"])
    overall_row = pd.DataFrame(
        [{
            "Channel": "OVERALL (weighted)",
            "n_test": result["n_test"].sum(),
            "MAPE": overall_mape,
            "Accuracy": 1 - overall_mape,
        }]
    )
    return pd.concat([result, overall_row], ignore_index=True)


def compare_baseline_vs_improved(
    baseline_eval: pd.DataFrame, improved_eval: pd.DataFrame
) -> pd.DataFrame:
    """Merges the two evaluation tables side by side for reporting."""
    merged = baseline_eval.merge(
        improved_eval, on="Channel", suffixes=("_baseline", "_improved")
    )
    merged["Accuracy_delta_pp"] = (
        merged["Accuracy_improved"] - merged["Accuracy_baseline"]
    ) * 100
    return merged[
        [
            "Channel",
            "n_test_baseline",
            "MAPE_baseline",
            "Accuracy_baseline",
            "MAPE_improved",
            "Accuracy_improved",
            "Accuracy_delta_pp",
        ]
    ].rename(columns={"n_test_baseline": "n_test"})


def evaluate_all_sku_channels(models: dict, test: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """
    Evaluates SKU-channel-level models (keyed by (product, channel)).
    Returns a per-SKU-channel + overall (volume-weighted) accuracy table.
    """
    feature_cols = feature_cols or FEATURE_COLS
    rows = []
    for (product, channel), model in models.items():
        sub = test[(test["Product"] == product) & (test["Channel"] == channel)]
        if len(sub) == 0:
            continue
        X, y_true = sub[feature_cols], sub[TARGET_COL].values
        y_pred = model.predict(X)
        metrics = evaluate_model(y_true, y_pred)
        rows.append({"Product": product, "Channel": channel, "n_test": len(sub), **metrics})

    result = pd.DataFrame(rows)
    overall_mape = np.average(result["MAPE"], weights=result["n_test"])
    overall_row = pd.DataFrame(
        [{
            "Product": "ALL",
            "Channel": "OVERALL (weighted)",
            "n_test": result["n_test"].sum(),
            "MAPE": overall_mape,
            "Accuracy": 1 - overall_mape,
        }]
    )
    return pd.concat([result, overall_row], ignore_index=True)


def aggregate_sku_eval_by_channel(sku_eval: pd.DataFrame) -> pd.DataFrame:
    """
    Rolls the per-SKU-channel evaluation up to the channel level, so it can
    be compared apples-to-apples against the pooled model's evaluate_all_channels()
    output (same test weeks, same weighting).
    """
    df = sku_eval[sku_eval["Product"] != "ALL"]
    rows = []
    for channel, sub in df.groupby("Channel"):
        mape = np.average(sub["MAPE"], weights=sub["n_test"])
        rows.append({"Channel": channel, "n_test": int(sub["n_test"].sum()), "MAPE": mape, "Accuracy": 1 - mape})
    result = pd.DataFrame(rows)
    overall_mape = np.average(result["MAPE"], weights=result["n_test"])
    overall_row = pd.DataFrame(
        [{"Channel": "OVERALL (weighted)", "n_test": int(result["n_test"].sum()),
          "MAPE": overall_mape, "Accuracy": 1 - overall_mape}]
    )
    return pd.concat([result, overall_row], ignore_index=True)


# ---------------------------------------------------------------------------
# 7. SHAP VALIDATION — Problem 2 (interpretability / forecast decomposition)
# ---------------------------------------------------------------------------

def price_sign_consistency(
    models: dict = None,
    df: pd.DataFrame = None,
    product: str = "Natural Juice 1L",
    channel: str = "Supermarkets",
    feature_cols: list = None,
    price_col: str = "Price_per_kg_USD",
    model=None,
) -> dict:
    """
    % of observations where SHAP attributes a POSITIVE contribution to
    the price feature (i.e. the wrong economic sign) for a given model.
    Works for either the raw price (price_col="Price_per_kg_USD") or the
    normalized index (price_col="Price_Index"), matching feature_cols.

    Pass either `models` (a {channel: model} dict, as returned by
    train_all_channels) OR `model` directly (e.g. a single SKU-channel
    model from train_all_sku_channels).
    """
    feature_cols = feature_cols or FEATURE_COLS
    subset = df[(df["Product"] == product) & (df["Channel"] == channel)]
    X = subset[feature_cols]

    if model is None:
        model = models[channel]
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    price_idx = feature_cols.index(price_col)
    price_shap = shap_values[:, price_idx]

    return {
        "pct_positive_sign": float((price_shap > 0).mean() * 100),
        "mean_shap": float(price_shap.mean()),
    }


def promo_vs_advertising_share(
    models: dict,
    df: pd.DataFrame,
    channel: str = "E-commerce",
    category: str = "Beverages",
) -> dict:
    """
    % share of total |SHAP| attributed to Promotion-related features vs
    Advertising Investment, for a given channel/category.
    """
    subset = df[(df["Channel"] == channel) & (df["Category"] == category)]
    X = subset[FEATURE_COLS]

    model = models[channel]
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    abs_total = np.abs(shap_values).sum()
    promo_idx = [
        FEATURE_COLS.index(c)
        for c in ("Promotion_Investment_USD", "Promo_Flag", "Interaction_Price_Promo")
    ]
    adv_idx = FEATURE_COLS.index("Advertising_Investment_USD")

    return {
        "promo_share_pct": float(np.abs(shap_values[:, promo_idx]).sum() / abs_total * 100),
        "advertising_share_pct": float(np.abs(shap_values[:, adv_idx]).sum() / abs_total * 100),
    }


def price_partial_dependence(
    model,
    df: pd.DataFrame,
    product: str = "Natural Juice 1L",
    channel: str = "Supermarkets",
    n_points: int = 20,
    feature_cols: list = None,
    price_col: str = "Price_per_kg_USD",
) -> pd.DataFrame:
    """
    Direct proof that the monotonic constraint holds: sweeps the price
    feature across its observed range (holding every other feature fixed
    at each row's actual values) and returns the AVERAGE predicted volume
    at each price point. Unlike the SHAP sign check, this must be
    non-increasing by construction when the model was trained with a
    -1 monotonic constraint on that feature.
    """
    feature_cols = feature_cols or FEATURE_COLS
    subset = df[(df["Product"] == product) & (df["Channel"] == channel)]
    X = subset[feature_cols].copy()

    price_min, price_max = X[price_col].min(), X[price_col].max()
    grid = np.linspace(price_min, price_max, n_points)

    avg_preds = []
    for p in grid:
        X_sweep = X.copy()
        X_sweep[price_col] = p
        avg_preds.append(model.predict(X_sweep).mean())

    return pd.DataFrame({price_col: grid, "Avg_Predicted_Volume": avg_preds})


def plot_partial_dependence(
    pdp_baseline: pd.DataFrame, pdp_improved: pd.DataFrame, out_path: str,
    price_col: str = "Price_per_kg_USD", x_label: str = "Price per kg (USD)",
    title_suffix: str = "", label_baseline: str = "Baseline (atual)",
    label_improved: str = "Improved (monotonic)",
):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(
        pdp_baseline[price_col], pdp_baseline["Avg_Predicted_Volume"],
        marker="o", label=label_baseline, color="#d62728",
    )
    ax.plot(
        pdp_improved[price_col], pdp_improved["Avg_Predicted_Volume"],
        marker="o", label=label_improved, color="#2ca02c",
    )
    ax.set_xlabel(x_label)
    ax.set_ylabel("Avg predicted volume (tons)")
    ax.set_title(f"Partial dependence: Price vs Predicted Volume{title_suffix}\nNatural Juice / Supermarkets")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 7B. RECURSIVE (MULTI-STEP) BACKTEST
# ---------------------------------------------------------------------------
#
# The evaluation above (evaluate_all_channels) is a ONE-STEP-AHEAD backtest:
# lag/rolling features in the test set are built from the REAL Sell_In_Tons,
# simulating a weekly reforecast where fresh actuals are always available.
#
# That is NOT what will happen for the 78-week future forecast: there is no
# real Sell_In there, so Lag_1w / Lag_4w / Rolling_Mean_4w / Rolling_Std_4w
# must be built from the model's OWN previous predictions, recursively. This
# section replays the same holdout period, but recursively, to measure how
# fast accuracy degrades as the forecast horizon grows -- exactly the
# behavior we'll see in the real 78-week forecast.

def recursive_backtest_sku(
    model,
    raw_df: pd.DataFrame,
    product: str,
    channel: str,
    feature_cols: list = None,
) -> pd.DataFrame:
    """
    Runs a true multi-step-ahead backtest for a single Product/Channel:
    walks forward week by week through the holdout period, and instead of
    using the real Sell_In_Tons to build Lag_1w/Lag_4w/Rolling_* (like the
    one-step-ahead backtest does), feeds the MODEL'S OWN prior predictions
    back in -- exactly what happens when forecasting 78 weeks with no real
    data arriving in between.

    `raw_df` must be the raw merged dataframe from load_data() (BEFORE
    engineer_features), so it still has one row per Product/Channel/Week
    with the untouched external variables and the real Sell_In_Tons (used
    only for computing accuracy afterwards, never fed into the features).
    """
    feature_cols = feature_cols or FEATURE_COLS
    sub = (
        raw_df[(raw_df["Product"] == product) & (raw_df["Channel"] == channel)]
        .sort_values("Week")
        .reset_index(drop=True)
    )

    cutoff = sub["Week"].quantile(HOLDOUT_QUANTILE, interpolation="nearest")
    train_idx = sub.index[sub["Week"] <= cutoff].tolist()
    test_idx = sub.index[sub["Week"] > cutoff].tolist()
    if len(test_idx) < 2 or len(train_idx) < 4:
        return pd.DataFrame()

    # History buffer starts as the REAL Sell_In for every training week
    # (this is legitimate "burn-in" -- those are real past actuals, known
    # at forecast time). From here on, only predictions get appended.
    history = sub.loc[train_idx, TARGET_COL].tolist()

    rows = []
    for horizon, idx in enumerate(test_idx, start=1):
        row = sub.loc[idx]

        promo_flag = int(pd.notna(row["Promotion_Type"]))
        prev_price = sub.loc[idx - 1, "Price_per_kg_USD"] if (idx - 1) in sub.index else np.nan
        price_change_pct = (
            (row["Price_per_kg_USD"] - prev_price) / prev_price
            if pd.notna(prev_price) and prev_price != 0 else np.nan
        )
        month = row["Week"].month

        feat = {
            "Price_per_kg_USD": row["Price_per_kg_USD"],
            "Numeric_Distribution": row["Numeric_Distribution"],
            "Weighted_Distribution": row["Weighted_Distribution"],
            "Advertising_Investment_USD": row["Advertising_Investment_USD"],
            "Promotion_Investment_USD": row["Promotion_Investment_USD"],
            "Promo_Flag": promo_flag,
            "Lag_1w": history[-1],
            "Lag_4w": history[-4] if len(history) >= 4 else history[0],
            "Rolling_Mean_4w": float(np.mean(history[-4:])),
            "Rolling_Std_4w": float(np.std(history[-4:], ddof=1)) if len(history[-4:]) > 1 else 0.0,
            "Price_Change_Pct": price_change_pct,
            "Month_Sin": np.sin(2 * np.pi * month / 12),
            "Month_Cos": np.cos(2 * np.pi * month / 12),
            "Holiday_Flag": 0,
            "Interaction_Price_Promo": row["Price_per_kg_USD"] * promo_flag,
        }
        X_row = pd.DataFrame([feat])[feature_cols]
        pred = float(model.predict(X_row)[0])

        rows.append({
            "Product": product,
            "Channel": channel,
            "Week": row["Week"],
            "Horizon": horizon,
            "Actual": row[TARGET_COL],
            "Predicted": pred,
        })

        # Feed the PREDICTION back -- never the real value -- simulating
        # no actuals arriving during the forecast window.
        history.append(pred)

    result = pd.DataFrame(rows)
    result["APE"] = (result["Actual"] - result["Predicted"]).abs() / result["Actual"]
    return result


def recursive_backtest_all(model_dict: dict, raw_df: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """Runs recursive_backtest_sku for every Product/Channel combination."""
    feature_cols = feature_cols or FEATURE_COLS
    all_results = []
    for channel, model in model_dict.items():
        products = raw_df.loc[raw_df["Channel"] == channel, "Product"].unique()
        for product in products:
            r = recursive_backtest_sku(model, raw_df, product, channel, feature_cols=feature_cols)
            if len(r):
                all_results.append(r)
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


def degradation_curve(recursive_results: pd.DataFrame) -> pd.DataFrame:
    """Aggregates recursive backtest results into Accuracy (1-MAPE) by horizon."""
    return (
        recursive_results.groupby("Horizon")
        .agg(Accuracy=("APE", lambda x: 1 - x.mean()), n=("APE", "size"))
        .reset_index()
    )


def plot_degradation_curve(curve_baseline: pd.DataFrame, curve_improved: pd.DataFrame, out_path: str):
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(curve_baseline["Horizon"], curve_baseline["Accuracy"] * 100, marker="o", label="Baseline", color="#d62728")
    ax.plot(curve_improved["Horizon"], curve_improved["Accuracy"] * 100, marker="o", label="Improved", color="#2ca02c")
    ax.set_xlabel("Horizonte (semanas à frente, sem dado real intermediário)")
    ax.set_ylabel("Accuracy (1 - MAPE) %")
    ax.set_title("Degradação da acurácia por horizonte\n(previsão recursiva / multi-step, todos os SKU-canal)")
    ax.axhline(75, color="gray", linestyle="--", linewidth=1, label="Benchmark (75%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 8. PLOTS
# ---------------------------------------------------------------------------

def plot_accuracy_comparison(comparison: pd.DataFrame, out_path: str):
    plot_df = comparison[comparison["Channel"] != "OVERALL (weighted)"]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(plot_df))
    width = 0.35
    ax.bar(x - width / 2, plot_df["Accuracy_baseline"] * 100, width, label="Baseline (atual)")
    ax.bar(x + width / 2, plot_df["Accuracy_improved"] * 100, width, label="Improved (regularizado + monotonic)")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Channel"])
    ax.set_ylabel("Accuracy (1 - MAPE) %")
    ax.set_title("Accuracy by channel — Baseline vs Improved")
    ax.axhline(75, color="gray", linestyle="--", linewidth=1, label="Industry benchmark (75%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_price_sign(sign_baseline: dict, sign_improved: dict, out_path: str):
    variants = ["Baseline", "Improved"]
    values = [sign_baseline["pct_positive_sign"], sign_improved["pct_positive_sign"]]
    fig, ax = plt.subplots(figsize=(5.5, 4))
    bars = ax.bar(variants, values, color=["#d62728", "#2ca02c"])
    ax.set_ylabel("% of weeks with WRONG-signed price SHAP (positive)")
    ax.set_title("Price elasticity sign consistency\nNatural Juice / Supermarkets")
    ax.set_ylim(0, 100)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + 2, f"{v:.0f}%", ha="center")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 9. MAIN
# ---------------------------------------------------------------------------

def main():
    outdir = Path(OUTDIR)
    outdir.mkdir(exist_ok=True, parents=True)

    # --- Load ---
    print("1) Loading raw data (Table 1 + Table 2)...")
    raw = load_data(DATA_PATH)
    print(f"   -> {len(raw)} rows loaded")

    # --- Feature engineering ---
    print("\n2) Engineering features (Table 4 rules)...")
    df = engineer_features(raw)
    print(f"   -> {len(df)} rows after feature engineering / dropna")

    # --- Split ---
    train, test = time_split(df)
    print(f"\n3) Time-based split: {len(train)} train rows / {len(test)} test rows")

    # --- Train ---
    print("\n4) Training BASELINE models (one per channel, current params)...")
    baseline_models = train_all_channels(train, BASELINE_PARAMS)

    print("5) Training IMPROVED models (one per channel, regularized + monotonic)...")
    improved_models = train_all_channels(train, IMPROVED_PARAMS)

    print("5b) Training IMPROVED_V2 models (also constrains Interaction_Price_Promo)...")
    improved_v2_models = train_all_channels(train, IMPROVED_V2_PARAMS)

    print("5c) Training IMPROVED_V3 models (SKU-relative Price_Index instead of raw price)...")
    train_norm, test_norm = add_price_index(train, test)
    improved_v3_models = train_all_channels(train_norm, IMPROVED_V3_PARAMS, feature_cols=FEATURE_COLS_NORM)

    print("5d) Training BASELINE_V3 models (Price_Index, but WITHOUT the monotonic constraint)...")
    # Same hyperparameters as BASELINE_PARAMS (no regularization, no constraint),
    # just fed the normalized price. Isolates: does normalization ALONE fix the
    # SHAP sign issue, or is the monotonic constraint also needed?
    baseline_v3_models = train_all_channels(train_norm, BASELINE_PARAMS, feature_cols=FEATURE_COLS_NORM)

    print("5e) Training SKU_LEVEL models (one per Product x Channel, no pooling)...")
    sku_level_models = train_all_sku_channels(train, SKU_LEVEL_PARAMS)

    # --- Evaluate: Accuracy = 1 - MAPE ---
    print("\n6) Evaluating BASELINE (Accuracy = 1 - MAPE)...")
    baseline_eval = evaluate_all_channels(baseline_models, test)
    print(baseline_eval.to_string(index=False))

    print("\n7) Evaluating IMPROVED (Accuracy = 1 - MAPE)...")
    improved_eval = evaluate_all_channels(improved_models, test)
    print(improved_eval.to_string(index=False))

    print("\n7b) Evaluating IMPROVED_V3 / Price_Index (Accuracy = 1 - MAPE)...")
    improved_v3_eval = evaluate_all_channels(improved_v3_models, test_norm, feature_cols=FEATURE_COLS_NORM)
    print(improved_v3_eval.to_string(index=False))

    print("\n7c) Evaluating BASELINE_V3 / Price_Index, no constraint (Accuracy = 1 - MAPE)...")
    baseline_v3_eval = evaluate_all_channels(baseline_v3_models, test_norm, feature_cols=FEATURE_COLS_NORM)
    print(baseline_v3_eval.to_string(index=False))

    print("\n7d) Evaluating SKU_LEVEL models (one per Product x Channel) -- per SKU-channel:")
    sku_level_eval = evaluate_all_sku_channels(sku_level_models, test)
    print(sku_level_eval.to_string(index=False))

    print("\n7e) SKU_LEVEL rolled up to channel level (apples-to-apples vs pooled models):")
    sku_level_by_channel = aggregate_sku_eval_by_channel(sku_level_eval)
    print(sku_level_by_channel.to_string(index=False))
    print("   (Compare this Accuracy column directly against Baseline/Improved in step 6/7 --")
    print("    same test weeks, same weighting, only difference is pooling vs no pooling.)")

    comparison = compare_baseline_vs_improved(baseline_eval, improved_eval)
    print("\n8) Baseline vs Improved comparison:")
    print(comparison.to_string(index=False))
    comparison.to_csv(outdir / "accuracy_comparison.csv", index=False)

    # --- SHAP validation ---
    print("\n9) Validating Problem 2 fix — price elasticity sign (Natural Juice / Supermarkets)...")
    full_df = pd.concat([train, test])
    sign_baseline = price_sign_consistency(baseline_models, full_df)
    sign_improved = price_sign_consistency(improved_models, full_df)
    sign_improved_v2 = price_sign_consistency(improved_v2_models, full_df)
    print(f"   Baseline:      {sign_baseline['pct_positive_sign']:.1f}% wrong-signed weeks, mean SHAP={sign_baseline['mean_shap']:.4f}")
    print(f"   Improved:      {sign_improved['pct_positive_sign']:.1f}% wrong-signed weeks, mean SHAP={sign_improved['mean_shap']:.4f}")
    print(f"   Improved_V2:   {sign_improved_v2['pct_positive_sign']:.1f}% wrong-signed weeks, mean SHAP={sign_improved_v2['mean_shap']:.4f}")
    print("   (V2 also constrains Interaction_Price_Promo -- tests the 'leakage through")
    print("    the interaction term' hypothesis. If V2's % drops a lot vs Improved, the")
    print("    hypothesis is confirmed.)")

    full_df_norm = pd.concat([train_norm, test_norm])
    sign_baseline_v3 = price_sign_consistency(
        baseline_v3_models, full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )
    sign_improved_v3 = price_sign_consistency(
        improved_v3_models, full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )
    print(f"   Baseline_V3 (normalization ONLY, no constraint): {sign_baseline_v3['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_baseline_v3['mean_shap']:.4f}")
    print(f"   Improved_V3 (normalization + constraint):        {sign_improved_v3['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_improved_v3['mean_shap']:.4f}")
    print("   (Comparing these two isolates whether normalization alone is enough, or")
    print("    whether the monotonic constraint is still doing real work on top of it.)")

    sku_model_nj_sup = sku_level_models.get(("Natural Juice 1L", "Supermarkets"))
    sign_sku_level = None
    if sku_model_nj_sup is not None:
        sign_sku_level = price_sign_consistency(
            df=full_df, product="Natural Juice 1L", channel="Supermarkets", model=sku_model_nj_sup
        )
        print(f"   SKU_LEVEL (no pooling, raw price, no constraint): {sign_sku_level['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_sku_level['mean_shap']:.4f}")
        print("   (Bonus check: does training per-SKU alone -- with no normalization and no")
        print("    monotonic constraint -- already fix the sign issue, since the model never")
        print("    sees other products' price scales in the first place?)")

    print("\n10) Validating Promotion vs Advertising attribution (E-commerce / Beverages)...")
    share_baseline = promo_vs_advertising_share(baseline_models, full_df)
    share_improved = promo_vs_advertising_share(improved_models, full_df)
    print(f"   Baseline: Promotion={share_baseline['promo_share_pct']:.1f}%  Advertising={share_baseline['advertising_share_pct']:.1f}%")
    print(f"   Improved: Promotion={share_improved['promo_share_pct']:.1f}%  Advertising={share_improved['advertising_share_pct']:.1f}%")

    print("\n11) Computing partial dependence (direct proof of the monotonic constraint)...")
    pdp_baseline = price_partial_dependence(baseline_models["Supermarkets"], full_df)
    pdp_improved = price_partial_dependence(improved_models["Supermarkets"], full_df)
    print("   Baseline PDP (should NOT be monotonic decreasing):")
    print(pdp_baseline.to_string(index=False))
    print("   Improved PDP (MUST be monotonic non-increasing):")
    print(pdp_improved.to_string(index=False))

    pdp_baseline_v3 = price_partial_dependence(
        baseline_v3_models["Supermarkets"], full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )
    pdp_improved_v3 = price_partial_dependence(
        improved_v3_models["Supermarkets"], full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )

    print("\n11b) Running RECURSIVE (multi-step) backtest -- simulates the 78-week")
    print("     future forecast, where lags come from predictions, not real data...")
    recursive_baseline = recursive_backtest_all(baseline_models, raw)
    recursive_improved = recursive_backtest_all(improved_models, raw)
    curve_baseline = degradation_curve(recursive_baseline)
    curve_improved = degradation_curve(recursive_improved)
    print("   Accuracy by horizon -- Baseline:")
    print(curve_baseline.to_string(index=False))
    print("   Accuracy by horizon -- Improved:")
    print(curve_improved.to_string(index=False))
    recursive_baseline.to_csv(outdir / "recursive_backtest_baseline.csv", index=False)
    recursive_improved.to_csv(outdir / "recursive_backtest_improved.csv", index=False)

    # --- Plots ---
    print("\n12) Generating plots...")
    plot_accuracy_comparison(comparison, str(outdir / "accuracy_by_channel.png"))
    plot_price_sign(sign_baseline, sign_improved, str(outdir / "price_sign_consistency.png"))
    plot_partial_dependence(pdp_baseline, pdp_improved, str(outdir / "price_partial_dependence.png"))
    plot_partial_dependence(
        pdp_baseline_v3, pdp_improved_v3, str(outdir / "price_index_partial_dependence.png"),
        price_col="Price_Index", x_label="Price_Index (1.0 = SKU's own average price)",
        title_suffix=" (normalized)",
        label_baseline="Normalized only (no constraint)",
        label_improved="Normalized + monotonic",
    )
    plot_degradation_curve(curve_baseline, curve_improved, str(outdir / "accuracy_degradation_by_horizon.png"))

    print(f"\nDone. Results saved to: {outdir.resolve()}")

    # Return everything so it can be inspected in a notebook, e.g.:
    #   results = main()
    #   print(results["sign_baseline"])
    return {
        "raw": raw,
        "df": df,
        "train": train,
        "test": test,
        "train_norm": train_norm,
        "test_norm": test_norm,
        "baseline_models": baseline_models,
        "improved_models": improved_models,
        "improved_v2_models": improved_v2_models,
        "improved_v3_models": improved_v3_models,
        "baseline_v3_models": baseline_v3_models,
        "sku_level_models": sku_level_models,
        "baseline_eval": baseline_eval,
        "improved_eval": improved_eval,
        "improved_v3_eval": improved_v3_eval,
        "baseline_v3_eval": baseline_v3_eval,
        "sku_level_eval": sku_level_eval,
        "sku_level_by_channel": sku_level_by_channel,
        "comparison": comparison,
        "sign_baseline": sign_baseline,
        "sign_improved": sign_improved,
        "sign_improved_v2": sign_improved_v2,
        "sign_improved_v3": sign_improved_v3,
        "sign_baseline_v3": sign_baseline_v3,
        "sign_sku_level": sign_sku_level,
        "share_baseline": share_baseline,
        "share_improved": share_improved,
        "pdp_baseline": pdp_baseline,
        "pdp_improved": pdp_improved,
        "pdp_improved_v3": pdp_improved_v3,
        "recursive_baseline": recursive_baseline,
        "recursive_improved": recursive_improved,
        "curve_baseline": curve_baseline,
        "curve_improved": curve_improved,
    }


if __name__ == "__main__":
    main()

1) Loading raw data (Table 1 + Table 2)...
   -> 2340 rows loaded

2) Engineering features (Table 4 rules)...
   -> 2280 rows after feature engineering / dropna

3) Time-based split: 1935 train rows / 345 test rows

4) Training BASELINE models (one per channel, current params)...
5) Training IMPROVED models (one per channel, regularized + monotonic)...
5b) Training IMPROVED_V2 models (also constrains Interaction_Price_Promo)...
5c) Training IMPROVED_V3 models (SKU-relative Price_Index instead of raw price)...
5d) Training BASELINE_V3 models (Price_Index, but WITHOUT the monotonic constraint)...
5e) Training SKU_LEVEL models (one per Product x Channel, no pooling)...

6) Evaluating BASELINE (Accuracy = 1 - MAPE)...
           Channel  n_test     MAPE  Accuracy
        E-commerce     115 0.803267  0.196733
      Supermarkets     115 0.287988  0.712012
       Traditional     115 0.363593  0.636407
OVERALL (weighted)     345 0.484949  0.515051

7) Evaluating IMPROVED (Accuracy = 1 - MAPE).